# Install Dependecies

In [1]:
%%capture
%pip install numpy
%pip install pandas
%pip install matplotlib.pyplot
%pip install python-terrier
%pip install gensim
%pip install "pyterrier-alpha[parallel]"

In [2]:
# Load java
!curl -s "https://get.sdkman.io" | bash && source "$HOME/.sdkman/bin/sdkman-init.sh" && sdk install java 11.0.22-amzn < /dev/null

# check java and version
!which java
!java -version
!readlink -f $(which java)
!ls -la /usr/lib/jvm
!java --version
!javac --version


                                -+syyyyyyys:
                            `/yho:`       -yd.
                         `/yh/`             +m.
                       .oho.                 hy                          .`
                     .sh/`                   :N`                `-/o`  `+dyyo:.
                   .yh:`                     `M-          `-/osysoym  :hs` `-+sys:      hhyssssssssy+
                 .sh:`                       `N:          ms/-``  yy.yh-      -hy.    `.N-````````+N.
               `od/`                         `N-       -/oM-      ddd+`     `sd:     hNNm        -N:
              :do`                           .M.       dMMM-     `ms.      /d+`     `NMMs       `do
            .yy-                             :N`    ```mMMM.      -      -hy.       /MMM:       yh
          `+d+`           `:/oo/`       `-/osyh/ossssssdNMM`           .sh:         yMMN`      /m.
         -dh-           :ymNMMMMy  `-/shmNm-`:N/-.``   `.sN            /N-         `NMMy      .m/
  

# Imports

In [3]:
import itertools
import json
import os
import re
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyterrier as pt
import gensim.downloader as api
from pathlib import Path
from tqdm.auto import tqdm
import pyterrier_alpha as pta
from pyterrier_alpha.parallel import PoolParallelTransformer

# PyTerrier - Local

In [ ]:
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# os.environ["JVM_PATH"] = "/usr/lib/jvm/java-17-openjdk-amd64/lib/server/libjvm.so"
# os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# if not pt.java.started():
#     pt.java.init()

# print("JAVA_HOME:", os.environ["JAVA_HOME"])
# print("JVM_PATH:", os.environ["JVM_PATH"])
# print("Java started:", pt.java.started())

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
JVM_PATH: /usr/lib/jvm/java-17-openjdk-amd64/lib/server/libjvm.so
Java started: True


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


# PyTerrier - Colab

In [5]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

if not pt.started():
    pt.init()

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Java started:", pt.java.started())

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


/tmp/ipykernel_519/631900236.py:4: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
Java started: True


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_519/631900236.py:5: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


# Load Dataset

In [10]:
# !cp -r /home/tlvj/msc_datalogi/2_semester/se/google_drive/ ./google_drive

In [ ]:
# print(os.getcwd())
# print(os.path.exists("./google_drive/indexes/full"))
# print(os.listdir("./google_drive/indexes/full"))

/home/tlvj
True
['data.lexicon.fsomaphash', 'build_time.json:Zone.Identifier', 'data.lexicon.fsomapfile:Zone.Identifier', 'data.lexicon.fsomaphash:Zone.Identifier', 'data.lexicon.fsomapfile', 'data.meta.idx', 'data.inverted.bf:Zone.Identifier', 'data.lexicon.fsomapid:Zone.Identifier', 'data.properties:Zone.Identifier', 'data.meta-0.fsomapfile:Zone.Identifier', 'build_time.json', 'data.document.fsarrayfile:Zone.Identifier', 'data.direct.bf:Zone.Identifier', 'data.properties', 'data.meta-0.fsomapfile', 'data.inverted.bf', 'data.meta.idx:Zone.Identifier', 'data.meta.zdata', 'data.direct.bf', 'data.document.fsarrayfile', 'data.meta.zdata:Zone.Identifier', 'data.lexicon.fsomapid']


In [6]:
base = 'https://raw.githubusercontent.com/Legenden84/search-engines/master/project_handout'

docs    = pd.read_json(f'{base}/docs2.jsonl', lines=True, dtype={'docno': str})
train_queries = pd.read_csv(f'{base}/train_queries.csv')
train_qrels   = pd.read_csv(f'{base}/train_qrels.csv')

# Load Indexes

In [7]:
index_path_none = "./indexes/full"
index_path_stop = "./indexes/stopwords"
index_path_stem = "./indexes/stemming"
index_path_stop_stem = "./indexes/stop-stem"

In [12]:
index_none = pt.IndexFactory.of(index_path_none)
index_stop = pt.IndexFactory.of(index_path_stop)
index_stem = pt.IndexFactory.of(index_path_stem)
index_stop_stem = pt.IndexFactory.of(index_path_stop_stem)

indices = {
    "stopwords": index_stop,
    "stop_stem": index_stop_stem,

    # Optional
    "none": index_none,
    "stem": index_stem
}

# Preprocessing

In [13]:
if "text" in train_queries.columns and "query" not in train_queries.columns:
    train_queries = train_queries.rename(columns={"text": "query"})

train_queries["qid"] = train_queries["qid"].astype(str)
train_queries["query"] = train_queries["query"].astype(str)

train_qrels["qid"] = train_qrels["qid"].astype(str)
train_qrels["docno"] = train_qrels["docno"].astype(str)

if "label" not in train_qrels.columns:
    if "relevance" in train_qrels.columns:
        train_qrels = train_qrels.rename(columns={"relevance": "label"})
    elif "rel" in train_qrels.columns:
        train_qrels = train_qrels.rename(columns={"rel": "label"})

train_qrels["label"] = train_qrels["label"].astype(int)

print("Training queries:")
display(train_queries.head())

print("Training qrels:")
display(train_qrels.head())

Training queries:


,qid,query
0,0,when did richmond last play in a preliminary f...
1,1,who sang what in the world's come over you
2,2,who produces the most wool in the world
3,3,where does alaska the last frontier take place
4,4,a day to remember all i want cameos


Training qrels:


,qid,docno,label,iteration
0,5743,D2,2,0
1,6266,D13,2,0
2,7918,D17,2,0
3,5215,D25,1,0
4,1173,D37,1,0


# Import tuning caches

In [14]:
cache_dir = Path("./results")
cache_dir.mkdir(parents=True, exist_ok=True)

bm25_cache_path = cache_dir / "bm25_tuning_results.json"
lm_cache_path   = cache_dir / "lm_tuning_results.json"
rm3_cache_path = cache_dir / "rm3_tuning_results.json"

tuning_results = []
best_models = {
    index_name: {}
    for index_name in indices.keys()
}

### Import BM25

In [33]:
def load_bm25_cache():
    """Returns (tuning_results, best_configs) from the cache file.
    ([], {}) if the cache does not exist yet."""
    if not bm25_cache_path.exists():
        return [], {}
    with open(bm25_cache_path, "r") as f:
        cache = json.load(f)
    return cache["tuning_results"], cache["best_configs"]

In [34]:
def sync_bm25_state_from_cache():
    """Refresh shared state (`tuning_results`, `best_models`) from the cache file.
    Idempotent."""
    cached_rows, cached_best = load_bm25_cache()

    # Replace BM25 rows in the shared tuning_results list.
    tuning_results[:] = [r for r in tuning_results if r.get("model") != "BM25"]
    tuning_results.extend(cached_rows)

    # Rebuild best_models[<index>]["BM25"] from the cache.
    for index_name, best_info in cached_best.items():
        if index_name not in indices:
            continue
        index  = indices[index_name]
        config = best_info["config"]
        best_models[index_name]["BM25"] = {
            "model": pt.terrier.Retriever(
                index,
                wmodel="BM25",
                controls={
                    "bm25.k_1": config["k1"],
                    "bm25.b":   config["b"]
                }
            ),
            "config": config,
            "score":  best_info["score"]
        }

In [35]:
if bm25_cache_path.exists():
    sync_bm25_state_from_cache()
    print(f"Loaded existing BM25 cache from {bm25_cache_path}")
else:
    print(
        f"No BM25 cache yet at {bm25_cache_path}.\n"
        f"Available indexes to tune: {list(indices.keys())}\n"
        f"Example:  tune_bm25_for_index('stopwords')"
    )

Loaded existing BM25 cache from results/bm25_tuning_results.json


### Import LM

In [36]:
def load_lm_cache():
    """Returns (tuning_results, best_configs) from the LM cache file.
    ([], {}) if the cache does not exist yet."""
    if not lm_cache_path.exists():
        return [], {}
    with open(lm_cache_path, "r") as f:
        cache = json.load(f)
    return cache["tuning_results"], cache["best_configs"]

In [37]:
def sync_lm_state_from_cache():
    """Refresh shared state (`tuning_results`, `best_models`) from the LM cache file.
    Idempotent."""
    cached_rows, cached_best = load_lm_cache()

    # Replace Hiemstra_LM rows in the shared tuning_results list.
    tuning_results[:] = [r for r in tuning_results if r.get("model") != "Hiemstra_LM"]
    tuning_results.extend(cached_rows)

    # Rebuild best_models[<index>]["Hiemstra_LM"] from the cache.
    for index_name, best_info in cached_best.items():
        if index_name not in indices:
            continue
        index  = indices[index_name]
        config = best_info["config"]
        best_models[index_name]["Hiemstra_LM"] = {
            "model": pt.terrier.Retriever(
                index,
                wmodel="Hiemstra_LM",
                controls={
                    "c": config["c"]
                }
            ),
            "config": config,
            "score":  best_info["score"]
        }

In [38]:
if lm_cache_path.exists():
    sync_lm_state_from_cache()
    print(f"Loaded existing LM cache from {lm_cache_path}")
else:
    print(
        f"No LM cache yet at {lm_cache_path}.\n"
        f"Available indexes to tune: {list(indices.keys())}\n"
        f"Example:  tune_lm_for_index('stopwords')"
    )

Loaded existing LM cache from results/lm_tuning_results.json


### Best model

In [39]:
EVAL_MEASURE = "ndcg_cut_10"

best_summary_rows = []

for index_name, model_dict in best_models.items():
    for model_name, info in model_dict.items():
        best_summary_rows.append({
            "index": index_name,
            "model": model_name,
            "best_config": info["config"],
            f"best_{EVAL_MEASURE}": round(info["score"], 4)
        })

best_summary_df = pd.DataFrame(best_summary_rows)

best_summary_df = best_summary_df.sort_values(
    by=f"best_{EVAL_MEASURE}",
    ascending=False
)

best_summary_df

,index,model,best_config,best_ndcg_cut_10
2,stop_stem,BM25,"{'k1': 0.9, 'b': 0.6}",0.4516
0,stopwords,BM25,"{'k1': 0.9, 'b': 0.6}",0.4434
3,stop_stem,Hiemstra_LM,{'c': 0.05},0.4411
1,stopwords,Hiemstra_LM,{'c': 0.05},0.4342


# 4 Relevance feedback and Query Expansion

The goal of this week is to add a pseudo relevance component to your search engine. You should use pseudo relevance feedback to expand the query and run it
with BM25 and LM. You should tune some of the parameters of pseudo relevance feedback using an evaluation measure of your choice, e.g., if you use RM3 pseudo
relevance feedback you can tune only fb terms and fb docs. You should tune with respect to one evaluation measure only (either MRR or NDCG@10). You should use your single best performing model-index combination as the backbone for the retrieval of the top-K documents to expand the queries and for the final retrieval.

In addition, you should use word embeddings to do query expansion as done by Kuzi et al. A popular word embedding choice is the word2vec pretrained on Google News corpus, but you can decide on another embedding. Another choice of embeddings is using contextualized word embeddings, such as BERT.

In [40]:
rm3_grid = {
    "fb_terms": [5, 10, 20, 30, 50],
    "fb_docs":  [3, 5, 10, 20],
}

SAMPLE_N = None

if SAMPLE_N is None:
    tune_queries = train_queries
    tune_qrels   = train_qrels
else:
    tune_queries = train_queries.sample(SAMPLE_N, random_state=42)
    tune_qrels   = train_qrels[train_qrels["qid"].isin(tune_queries["qid"])]

print(f"Tuning on {len(tune_queries)} queries / {len(tune_qrels)} qrels")

Tuning on 10000 queries / 17746 qrels


In [41]:
# ---------------------------------------------------------------
# Pseudo-Relevance Feedback (Week 4)
# Backbone = single best (index, model) combination from above.
# ---------------------------------------------------------------
prf_best_row = best_summary_df.iloc[0]
BACKBONE_INDEX_NAME = prf_best_row["index"]
BACKBONE_MODEL_NAME = prf_best_row["model"]
BACKBONE_CONFIG     = prf_best_row["best_config"]

backbone_index = indices[BACKBONE_INDEX_NAME]

if BACKBONE_MODEL_NAME == "BM25":
    backbone_retriever = pt.terrier.Retriever(
        backbone_index,
        wmodel="BM25",
        controls={"bm25.k_1": BACKBONE_CONFIG["k1"],
                  "bm25.b":   BACKBONE_CONFIG["b"]},
        metadata=["docno", "text"]
    )
else:
    backbone_retriever = pt.terrier.Retriever(
        backbone_index,
        wmodel="Hiemstra_LM",
        controls={"c": BACKBONE_CONFIG["c"]},
        metadata=["docno", "text"]
    )

print(f"PRF backbone: index={BACKBONE_INDEX_NAME}, model={BACKBONE_MODEL_NAME}, config={BACKBONE_CONFIG}")

PRF backbone: index=stop_stem, model=BM25, config={'k1': 0.9, 'b': 0.6}


In [42]:
def build_rm3_pipeline(fb_terms, fb_docs, first, second, index):
    rm3 = pt.rewrite.RM3(index, fb_terms=fb_terms, fb_docs=fb_docs)
    return first >> rm3 >> second

In [43]:
rm3_cache_path = cache_dir / "rm3_tuning_results.json"

# --- Dedicated retrievers for RM3 -----------------------------------
# First stage: only need top-max(fb_docs) docs, no text metadata
#              (pt.rewrite.RM3 reads doc terms from the index directly).
# Second stage: standard retriever for the final ranking.
MAX_FB_DOCS = max(rm3_grid["fb_docs"])

if BACKBONE_MODEL_NAME == "BM25":
    _ctrls = {"bm25.k_1": BACKBONE_CONFIG["k1"], "bm25.b": BACKBONE_CONFIG["b"]}
    _wmodel = "BM25"
else:
    _ctrls = {"c": BACKBONE_CONFIG["c"]}
    _wmodel = "Hiemstra_LM"

backbone_first = pt.terrier.Retriever(
    backbone_index, wmodel=_wmodel, controls=_ctrls, num_results=MAX_FB_DOCS,
)
backbone_second = pt.terrier.Retriever(
    backbone_index, wmodel=_wmodel, controls=_ctrls,
)

def tune_rm3(force=False):
    """Tune RM3 (fb_terms, fb_docs); select on sample, score winner on full set."""
    if rm3_cache_path.exists() and not force:
        with open(rm3_cache_path) as f:
            cache = json.load(f)
        print(f"Loaded RM3 cache: best={cache['best_config']} "
              f"{EVAL_MEASURE}={cache['best_score']:.4f}")
        return cache

    configs = list(itertools.product(rm3_grid["fb_terms"], rm3_grid["fb_docs"]))
    total = len(configs)
    rows = []
    best_score, best_cfg = -1, None

    t0 = time.time()
    for step, (ft, fd) in enumerate(configs, 1):
        ts = time.time()
        pipeline = build_rm3_pipeline(ft, fd, backbone_first, backbone_second, backbone_index)
        exp = pt.Experiment(
            [pipeline], tune_queries, tune_qrels,
            eval_metrics=[EVAL_MEASURE], filter_by_qrels=True,
        )
        score = float(exp[EVAL_MEASURE].iloc[0])
        rows.append({"fb_terms": ft, "fb_docs": fd,
                     f"sample_{EVAL_MEASURE}": score})
        print(f"[{step:>2}/{total}] fb_terms={ft:>2} fb_docs={fd:>2}  "
              f"sample_{EVAL_MEASURE}={score:.4f}  ({time.time()-ts:.1f}s)",
              flush=True)
        if score > best_score:
            best_score, best_cfg = score, {"fb_terms": ft, "fb_docs": fd}

    print(f"Sample tuning done in {time.time()-t0:.1f}s.  "
          f"Best on sample: {best_cfg} ({best_score:.4f})")

    # Refit winner on full training set for an honest reported number.
    final_pipeline = build_rm3_pipeline(
        best_cfg["fb_terms"], best_cfg["fb_docs"],
        backbone_first, backbone_second, backbone_index,
    )
    final_exp = pt.Experiment(
        [final_pipeline], train_queries, train_qrels,
        eval_metrics=[EVAL_MEASURE], filter_by_qrels=True,
    )
    full_score = float(final_exp[EVAL_MEASURE].iloc[0])
    print(f"Best on full set: {best_cfg} -> {EVAL_MEASURE}={full_score:.4f}")

    cache = {
        "backbone": {
            "index": BACKBONE_INDEX_NAME,
            "model": BACKBONE_MODEL_NAME,
            "config": BACKBONE_CONFIG,
        },
        "eval_measure":   EVAL_MEASURE,
        "grid":           rm3_grid,
        "sample_size":    SAMPLE_N,
        "tuning_results": rows,
        "best_config":    best_cfg,
        "best_sample_score": best_score,
        "best_score":     full_score,
    }
    with open(rm3_cache_path, "w") as f:
        json.dump(cache, f, indent=4)
    print(f"Saved RM3 tuning to {rm3_cache_path}")
    return cache